# **PIANO AMT (Onsets & Frames) — Demo**



# Setup

In [ ]:
#@title 0. Configuration
REPO_URL = "https://github.com/Mobinmo83/AMT.git"
REPO_BRANCH = "main"
REPO_CLONE_DIR = "/content/repo"

HF_REPO_ID_OVERRIDE = "Mobinmo83/piano-amt-demo"
HF_CHECKPOINT_FILENAME_OVERRIDE = "checkpoints/best.pt"
MODEL_COMPLEXITY_OVERRIDE = 48

In [ ]:
#@title 1. Cloning repo and installing public demo requirements
import os
import shutil
from pathlib import Path

repo_dir = Path(REPO_CLONE_DIR)
if repo_dir.exists():
    shutil.rmtree(repo_dir)

!git clone -q --branch "$REPO_BRANCH" "$REPO_URL" "$REPO_CLONE_DIR"
%cd $REPO_CLONE_DIR

# FluidSynth + GM soundfont improve predicted MIDI playback in Colab.
!apt-get -qq update
!apt-get -qq install -y fluidsynth fluid-soundfont-gm > /dev/null
!pip -q install -r requirements_demo.txt huggingface_hub


In [ ]:
#@title 2. Imports and runtime setup
import json
import os
import sys
import uuid
import shutil
from pathlib import Path

import pandas as pd
import pretty_midi
import torch
import ipywidgets as widgets
from google.colab import files
from huggingface_hub import hf_hub_download
from IPython.display import Audio, FileLink, HTML, Markdown, display

PROJECT_ROOT = REPO_CLONE_DIR

os.environ["AMT_DEMO_HF_REPO_ID"] = HF_REPO_ID_OVERRIDE
os.environ["AMT_DEMO_HF_CHECKPOINT_FILENAME"] = HF_CHECKPOINT_FILENAME_OVERRIDE
os.environ["AMT_DEMO_MODEL_COMPLEXITY"] = str(MODEL_COMPLEXITY_OVERRIDE)
os.environ["AMT_DEMO_REPO_ROOT"] = PROJECT_ROOT

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from demo.checkpoint import load_demo_model, format_checkpoint_summary, maybe_torchinfo_summary
from demo.decoder_presets import (
    DEFAULT_MODE,
    config_table_dict,
    get_decoder_preset,
    list_decoder_modes,
    make_decoder_config,
)
from demo.demo_config import AUDIO_DIR, HTML_DIR, MIDI_DIR, PLOT_DIR, ensure_demo_dirs
from demo.evaluation import (
    compare_decoder_modes,
    decoder_config_table,
    evaluate_prediction_vs_gt,
    main_scores_table,
    supplementary_table,
)
from demo.inference import (
    gt_rolls_to_note_events,
    prediction_to_note_events,
    run_model_on_mel,
    save_gt_eval_midi,
    save_prediction_midi,
)
from demo.rendering import (
    PIANO_PROGRAMS,
    plot_midi_with_sustain_and_velocity,
    plot_note_events_colored,
    plot_piano_roll_side_by_side,
    plot_pred_vs_gt_events,
    plot_roll_diff,
    render_visual_midi,
    synthesize_and_save,
    synthesize_pretty_midi,
)
from demo.sources import (
    audio_path_to_mel,
    list_demo_sample_names,
    load_ground_truth_labels,
    resolve_demo_sample_paths,
    save_uploaded_audio,
)

ensure_demo_dirs()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
#@title 3. Download demo assets from Hugging Face
PROJECT_ROOT_PATH = Path(PROJECT_ROOT)
LOCAL_MANIFEST = PROJECT_ROOT_PATH / "demo" / "sample_manifest.json"

manifest_cache_path = hf_hub_download(
    repo_id=HF_REPO_ID_OVERRIDE,
    filename="demo/sample_manifest.json",
)
LOCAL_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(manifest_cache_path, LOCAL_MANIFEST)

with open(LOCAL_MANIFEST, "r", encoding="utf-8") as f:
    manifest = json.load(f)

# Download the public demo assets used by the notebook.  The original MIDI is
# included for listening/display/download; metric ground truth still comes from
# the cached label rolls to match the final evaluation notebook.
for sample in manifest["samples"]:
    for key in ["audio", "labels", "midi", "original_midi"]:
        remote_path = sample.get(key)
        if not remote_path:
            continue
        cached_file = hf_hub_download(
            repo_id=HF_REPO_ID_OVERRIDE,
            filename=remote_path,
        )
        local_target = PROJECT_ROOT_PATH / remote_path
        local_target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(cached_file, local_target)

print(f"Downloaded manifest and {len(manifest['samples'])} public demo sample(s).")


# Initialize Model

In [ ]:

#@title 4. Loading checkpoint
MODEL, CKPT, CKPT_PATH = load_demo_model(DEVICE)


In [ ]:
#@title 5. Model info
Model_info = False  #@param {type:"boolean"}

if Model_info:
    print(format_checkpoint_summary(MODEL, CKPT, CKPT_PATH))
    print("Compact torchinfo summary:")
    print(maybe_torchinfo_summary(MODEL))
    print("Full architecture:")
    print(MODEL)

# Upload audio

In [ ]:

#@title 6. Upload Audio or choose from test set
DEMO_SAMPLE_NAMES = list_demo_sample_names()
SOURCE_MODE = "Upload custom audio" #@param ["Public demo sample", "Upload custom audio"]
DEMO_SAMPLE_INDEX = 0 #@param {type:"integer"}

print("Source mode:", SOURCE_MODE)
if SOURCE_MODE == "Public demo sample":
    if not DEMO_SAMPLE_NAMES:
        raise RuntimeError("No public demo samples were found.")
    if DEMO_SAMPLE_INDEX < 0 or DEMO_SAMPLE_INDEX >= len(DEMO_SAMPLE_NAMES):
        raise IndexError(f"DEMO_SAMPLE_INDEX must be between 0 and {len(DEMO_SAMPLE_NAMES)-1}")
    print("Selected sample:", DEMO_SAMPLE_NAMES[DEMO_SAMPLE_INDEX])
else:
    print("You will upload an audio file in the next cell.")




In [ ]:
#@title 7. Preparing input audio
from pathlib import Path

original_midi_path = None
metadata = {}

if SOURCE_MODE == "Public demo sample":
    selected_name = DEMO_SAMPLE_NAMES[DEMO_SAMPLE_INDEX]
    audio_path, labels_path, original_midi_path, metadata = resolve_demo_sample_paths(selected_name, PROJECT_ROOT)

    mel = audio_path_to_mel(audio_path)
    gt = load_ground_truth_labels(labels_path)
    title = Path(audio_path).stem

    source_type = "demo_sample"
    print("Prepared public demo sample:", title)
    if original_midi_path is not None:
        print("Original MAESTRO MIDI:", original_midi_path)
else:
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("Please upload an audio file.")

    first_name = next(iter(uploaded.keys()))
    audio_path = save_uploaded_audio(uploaded[first_name], first_name)
    mel = audio_path_to_mel(audio_path)
    gt = None
    title = Path(audio_path).stem

    source_type = "custom_upload"
    print("Prepared uploaded audio:", title)

print("Audio path:", audio_path)
print("Mel shape:", tuple(mel.shape))
if metadata:
    display(pd.DataFrame([metadata]).T.rename(columns={0: "Value"}))


In [ ]:
#@title 8. transcription and MIDI decode
# Choose the final tuned modes or turn on custom values to demonstrate parameter effects.
DECODER_MODE = "quality_m2_m3_m4" #@param ["quality_m2_m3_m4", "efficient_m3_m4", "baseline", "m2_only", "m3_only", "m4_only"]
PIANO_SOUND = "Acoustic Grand" #@param ["Acoustic Grand", "Bright Acoustic", "Electric Grand", "Honky-tonk", "Electric Piano"]

USE_CUSTOM_DECODER_VALUES = False #@param {type:"boolean"}
CUSTOM_ONSET_THRESHOLD = 0.40 #@param {type:"number"}
CUSTOM_FRAME_THRESHOLD = 0.40 #@param {type:"number"}
CUSTOM_OFFSET_THRESHOLD = 0.50 #@param {type:"number"}

CUSTOM_USE_ONSET_CONDITIONED_OFFSET = False #@param {type:"boolean"}
CUSTOM_USE_FRAME_SMOOTHING = True #@param {type:"boolean"}
CUSTOM_FRAME_SMOOTHING_KERNEL = 3 #@param {type:"integer"}
CUSTOM_FRAME_SMOOTHING_METHOD = "closing" #@param ["closing", "median", "gaussian"]

CUSTOM_MIN_NOTE_DURATION_MS = 55.0 #@param {type:"number"}
CUSTOM_USE_DUPLICATE_REMOVAL = True #@param {type:"boolean"}
CUSTOM_DUPLICATE_TOLERANCE_SEC = 0.06 #@param {type:"number"}

CUSTOM_USE_CHORD_GROUPING = False #@param {type:"boolean"}
CUSTOM_CHORD_TOLERANCE_SEC = 0.03 #@param {type:"number"}
CUSTOM_CHORD_SNAP_TO = "median" #@param ["median", "first"]

CUSTOM_USE_ADAPTIVE_THRESHOLDS = False #@param {type:"boolean"}
CUSTOM_ADAPTIVE_ONSET_K = 0.5 #@param {type:"number"}
CUSTOM_ADAPTIVE_FRAME_K = 0.5 #@param {type:"number"}

CUSTOM_USE_PEDAL_EXTENSION = False #@param {type:"boolean"}
CUSTOM_PEDAL_ENERGY_THRESHOLD = 10.0 #@param {type:"number"}
CUSTOM_PEDAL_MAX_EXTENSION_SEC = 2.0 #@param {type:"number"}

if USE_CUSTOM_DECODER_VALUES:
    cfg = make_decoder_config(
        DECODER_MODE,
        onset_threshold=float(CUSTOM_ONSET_THRESHOLD),
        frame_threshold=float(CUSTOM_FRAME_THRESHOLD),
        offset_threshold=float(CUSTOM_OFFSET_THRESHOLD),
        use_onset_conditioned_offset=bool(CUSTOM_USE_ONSET_CONDITIONED_OFFSET),
        use_frame_smoothing=bool(CUSTOM_USE_FRAME_SMOOTHING),
        frame_smoothing_kernel=int(CUSTOM_FRAME_SMOOTHING_KERNEL),
        frame_smoothing_method=str(CUSTOM_FRAME_SMOOTHING_METHOD),
        min_note_duration_ms=float(CUSTOM_MIN_NOTE_DURATION_MS),
        use_duplicate_removal=bool(CUSTOM_USE_DUPLICATE_REMOVAL),
        duplicate_tolerance_sec=float(CUSTOM_DUPLICATE_TOLERANCE_SEC),
        use_chord_grouping=bool(CUSTOM_USE_CHORD_GROUPING),
        chord_tolerance_sec=float(CUSTOM_CHORD_TOLERANCE_SEC),
        chord_snap_to=str(CUSTOM_CHORD_SNAP_TO),
        use_adaptive_thresholds=bool(CUSTOM_USE_ADAPTIVE_THRESHOLDS),
        adaptive_onset_k=float(CUSTOM_ADAPTIVE_ONSET_K),
        adaptive_frame_k=float(CUSTOM_ADAPTIVE_FRAME_K),
        use_pedal_extension=bool(CUSTOM_USE_PEDAL_EXTENSION),
        pedal_energy_threshold=float(CUSTOM_PEDAL_ENERGY_THRESHOLD),
        pedal_max_extension_sec=float(CUSTOM_PEDAL_MAX_EXTENSION_SEC),
    )
else:
    cfg = get_decoder_preset(DECODER_MODE)

display(Markdown("### Active decoder configuration"))
display(decoder_config_table(cfg))

pred = run_model_on_mel(MODEL, mel, DEVICE)
print("Prediction tensor shapes:")
for k, v in pred.items():
    print(f"  {k}: {tuple(v.shape)}")

run_id = f"{title}_{cfg.name}_{uuid.uuid4().hex[:8]}"

pred_midi_path = MIDI_DIR / f"{run_id}_predicted.mid"
pred_wav_path = AUDIO_DIR / f"{run_id}_predicted.wav"
pred_png_path = PLOT_DIR / f"{run_id}_pred_roll.png"
pred_events_png_path = PLOT_DIR / f"{run_id}_pred_events.png"
pred_html_path = HTML_DIR / f"{run_id}_pred_visual_midi.html"

gt_eval_midi_path = None
gt_eval_wav_path = None
gt_png_path = None
diff_png_path = None
event_overlay_path = None
original_midi_wav_path = None
original_sustain_png_path = None
original_html_path = None

program = PIANO_PROGRAMS.get(PIANO_SOUND, 0)
pred_midi_path = save_prediction_midi(pred, pred_midi_path, decoder_config=cfg, program=program)
pred_pm = pretty_midi.PrettyMIDI(str(pred_midi_path))
pred_wav_path = synthesize_and_save(pred_pm, pred_wav_path, piano_sound=PIANO_SOUND)
render_visual_midi(pred_pm, html_path=pred_html_path, show_inline=False)

if gt is not None:
    gt_eval_midi_path = MIDI_DIR / f"{run_id}_ground_truth_eval_rolls.mid"
    gt_eval_wav_path = AUDIO_DIR / f"{run_id}_ground_truth_eval_rolls.wav"
    gt_png_path = PLOT_DIR / f"{run_id}_ground_truth_and_pred_rolls.png"
    diff_png_path = PLOT_DIR / f"{run_id}_diff_roll.png"
    event_overlay_path = PLOT_DIR / f"{run_id}_pred_vs_gt_events.png"
    save_gt_eval_midi(gt, gt_eval_midi_path, program=program)
    gt_eval_pm = pretty_midi.PrettyMIDI(str(gt_eval_midi_path))
    synthesize_and_save(gt_eval_pm, gt_eval_wav_path, piano_sound=PIANO_SOUND)

if original_midi_path is not None and Path(original_midi_path).exists():
    original_pm = pretty_midi.PrettyMIDI(str(original_midi_path))
    original_midi_wav_path = AUDIO_DIR / f"{run_id}_original_maestro_midi.wav"
    original_sustain_png_path = PLOT_DIR / f"{run_id}_original_velocity_sustain.png"
    original_html_path = HTML_DIR / f"{run_id}_original_visual_midi.html"
    synthesize_and_save(original_pm, original_midi_wav_path, piano_sound=PIANO_SOUND)
    render_visual_midi(original_pm, html_path=original_html_path, show_inline=False)

print("Saved predicted MIDI:", pred_midi_path)
print("Saved predicted WAV:", pred_wav_path)


# Prediction and Inference


In [ ]:
#@title 9. Prediction results
import matplotlib.pyplot as plt

display(Markdown(f"## Results — {title}"))
display(Markdown(f"**Source:** `{source_type}`"))
display(Markdown(f"**Decoder mode:** `{DECODER_MODE}` → `{cfg.name}`"))
display(Markdown(f"**Piano sound:** `{PIANO_SOUND}`"))

display(Markdown("### Original audio"))
display(Audio(filename=str(audio_path)))

display(Markdown("### Predicted MIDI audio"))
display(Audio(filename=str(pred_wav_path)))

display(Markdown("### Predicted MIDI note events"))
pred_events = prediction_to_note_events(pred, decoder_config=cfg)
pred_events_fig = plot_note_events_colored(
    pred_events,
    title=f"Predicted note events — {cfg.name}",
    save_path=pred_events_png_path,
)
display(pred_events_fig)
plt.close(pred_events_fig)

display(Markdown("### Predicted piano roll"))
pred_fig = plot_piano_roll_side_by_side(
    pred_frame=pred["frame"],
    gt_frame=None,
    title=f"Prediction — {title}",
    save_path=pred_png_path,
    frame_threshold=cfg.frame_threshold,
)
display(pred_fig)
plt.close(pred_fig)

display(Markdown("### Predicted Visual MIDI"))
pred_visual = render_visual_midi(pred_pm, html_path=pred_html_path, show_inline=True)
if pred_visual is None:
    display(Markdown(f"Visual MIDI HTML saved here: `{pred_html_path}`"))


In [ ]:
#@title 10. Original audio and evaluation (test-set demo samples only)
import matplotlib.pyplot as plt

if gt is None:
    display(Markdown("> Uploaded custom audio has no ground-truth label rolls, so evaluation and original MIDI comparison are skipped."))
else:
    display(Markdown("### Evaluation protocol"))
    display(Markdown(
        "Metrics use the cached ground-truth label rolls decoded with the same GT path used in the final validation/test evaluation. "
        "The original MAESTRO MIDI is displayed for listening and download, but it is not used as the metric reference."
    ))

    if original_midi_path is not None and Path(original_midi_path).exists():
        display(Markdown("### Original MAESTRO MIDI audio"))
        display(Audio(filename=str(original_midi_wav_path)))

        display(Markdown("### Original MAESTRO MIDI: velocity and sustain"))
        original_fig = plot_midi_with_sustain_and_velocity(
            original_midi_path,
            title=f"Original MAESTRO MIDI — {title}",
            save_path=original_sustain_png_path,
        )
        display(original_fig)
        plt.close(original_fig)

        display(Markdown("### Original MAESTRO Visual MIDI"))
        original_visual = render_visual_midi(original_pm, html_path=original_html_path, show_inline=True)
        if original_visual is None:
            display(Markdown(f"Original Visual MIDI HTML saved here: `{original_html_path}`"))

    display(Markdown("### Evaluation ground-truth MIDI reconstructed from cached rolls"))
    display(Audio(filename=str(gt_eval_wav_path)))

    results = evaluate_prediction_vs_gt(pred, gt, decoder_config=cfg)

    display(Markdown("### Predicted vs evaluation ground-truth piano rolls"))
    comp_fig = plot_piano_roll_side_by_side(
        pred_frame=pred["frame"],
        gt_frame=gt["frame"],
        title=title,
        save_path=gt_png_path,
        frame_threshold=cfg.frame_threshold,
    )
    display(comp_fig)
    plt.close(comp_fig)

    diff_fig = plot_roll_diff(
        pred_frame=pred["frame"],
        gt_frame=gt["frame"],
        frame_threshold=cfg.frame_threshold,
        save_path=diff_png_path,
    )
    display(diff_fig)
    plt.close(diff_fig)

    display(Markdown("### Predicted vs evaluation ground-truth note events"))
    overlay_fig = plot_pred_vs_gt_events(
        results["events"]["pred"],
        results["events"]["gt_eval"],
        title=f"Prediction vs GT eval-roll events — {cfg.name}",
        max_time=60,
        save_path=event_overlay_path,
    )
    display(overlay_fig)
    plt.close(overlay_fig)

    display(Markdown("### Quantitative evaluation"))
    display(main_scores_table(results).style.format({
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1": "{:.4f}",
    }))

    display(Markdown("### Supplementary / failure-case analysis"))
    display(supplementary_table(results).style.format({"Value": "{:.4f}"}))

    display(Markdown("### Baseline vs efficient vs quality decoder comparison"))
    comparison = compare_decoder_modes(pred, gt, modes=["baseline", "efficient_m3_m4", "quality_m2_m3_m4"])
    display(comparison.style.format({
        "Frame F1": "{:.4f}",
        "Note F1": "{:.4f}",
        "N+Off F1": "{:.4f}",
        "N+Off+Vel F1": "{:.4f}",
    }))


In [ ]:
#@title 11 downloading files # add orginal audio and predicted audio downlaod as well
print("Download files")

download_items = [
    ("Predicted selected-mode MIDI", pred_midi_path),
    ("Predicted selected-mode audio WAV", pred_wav_path),
    ("Predicted piano-roll PNG", pred_png_path),
    ("Predicted note-event PNG", pred_events_png_path),
    ("Predicted Visual MIDI HTML", pred_html_path),
    ("Original input audio", audio_path),
]

if gt is not None:
    download_items.extend([
        ("Evaluation GT MIDI from cached rolls", gt_eval_midi_path),
        ("Evaluation GT MIDI audio WAV", gt_eval_wav_path),
        ("Prediction vs GT piano-roll PNG", gt_png_path),
        ("Frame diff PNG", diff_png_path),
        ("Predicted vs GT event overlay PNG", event_overlay_path),
    ])

if original_midi_path is not None and Path(original_midi_path).exists():
    download_items.extend([
        ("Original MAESTRO MIDI", original_midi_path),
        ("Original MAESTRO MIDI audio WAV", original_midi_wav_path),
        ("Original MIDI velocity+sustain PNG", original_sustain_png_path),
        ("Original Visual MIDI HTML", original_html_path),
    ])

buttons = []
for label, path in download_items:
    if path is None:
        continue
    path = Path(path)
    if not path.exists():
        continue
    btn = widgets.Button(description=label[:38], icon="download", layout=widgets.Layout(width="360px"))
    def _make_handler(p):
        def _handler(_):
            files.download(str(p))
        return _handler
    btn.on_click(_make_handler(path))
    buttons.append(btn)

if buttons:
    display(widgets.VBox(buttons))
else:
    print("No downloadable files found yet. Run the previous cells first.")
